<a href="https://colab.research.google.com/github/chunhoseo/HappyWorld/blob/main/DL04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
# 1. 파이썬 로 XOR 문제 풀기

X = np.array([[0,0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])


In [ ]:
# 활성화 함수
# sigmoid 함수
# 입력값을 0 ~ 1 사이의 값으로 변환한다.
#                   1
# sigmoid  =    ----------
#                1 + e^-x

def sigmoid(x):
  return 1 / (1+np.exp(-x))


# sigmoid 함수의 미분
# 역전파(Backpropagation)  에서 사용
# sigmoid'(x) = sigmoid(x) *(1-sigmoid(x))

def sigmoid_derivation(x):
  return x  * (1-x)

# 가중치의 초기화
# 실행할때마다 난수를생성 , 교육용으로 결과를 재현하기 위해 사용
np.random.seed(1)

# 입력 뉴런 2개 가 은닉 뉴런 4개와 연결
# 가중치 개수: 2*4 = 8

# W1의 shape = (2, 4)
W1 = np.random.uniform(-1,1, (2,4))
W1

array([[-0.16595599,  0.44064899, -0.99977125, -0.39533485],
       [-0.70648822, -0.81532281, -0.62747958, -0.30887855]])

In [ ]:
# 은닉층 뉴런이 4개 ==> bias : 4
b1 = np.zeros((1,4))
b1

array([[0., 0., 0., 0.]])

In [ ]:
# 출력층 뉴런 1개   W2 4행 1열 짜리 1
#   bias  1개
W2 = np.random.uniform(-1, 1, (4, 1))
W2

array([[-0.20646505],
       [ 0.07763347],
       [-0.16161097],
       [ 0.370439  ]])

In [ ]:
b2 = np.zeros((1,1))
b2

array([[0.]])

In [ ]:
# 학습률
# 가중치를 한번 학습할때 얼마나 많이 변경할것인지 결정
learning_rate = 0.5



In [ ]:
for epoch in range(10000):
  # 순전파
  # z = X*W + b
  hidden_input = np.dot(X, W1) + b1

  # 계산된 값을 sigmoid 함수에 통과
  hidden_output = sigmoid(hidden_input)

  # 은닉층 ==> 출력층
  final_input = np.dot(hidden_output, W2) + b2

  # 출력층 에도 sigmoid 함수를 적용
  output = sigmoid(final_input)

  # 오차 계산
  # 실제 정답 - 예측값

  # 정답 1
  # 예측 0.7
  # 오차 : 0.3
  error = y - output

  # 출력층이 얼마나 잘못되었는지 계산
  # 오차 * sigmoid 미분값
  output_delta = (error * sigmoid_derivation(output))

  # 은닉층으로 오차 전달
  # W2 가 (4, 1)이므로 전치하면 (1, 4)
  hidden_error = (np.dot(output_delta, W2.T))

  #은닉층에서 sigmoid의 미분값을 적용
  hidden_delta = ( hidden_error* sigmoid_derivation(hidden_output))

  # 가중치 , bias 가중치 수정
  W2 += (learning_rate*np.dot(hidden_output.T, output_delta))

  b2 += (learning_rate* np.sum(output_delta, axis=0, keepdims=True))

  W1 += (learning_rate* np.dot(X.T, hidden_delta))

  b1 += (learning_rate* np.sum(hidden_delta, axis=0, keepdims=True))



In [ ]:
# 학습 결과 확인
print("최종출력")
print(np.round(output, 3))

최종출력
[[0.015]
 [0.981]
 [0.988]
 [0.02 ]]


In [ ]:
# 0.5 이상이면 1
# 이하면 0
print("XOR 최종 결과")
print((output>=0.5).astype(int))

XOR 최종 결과
[[0]
 [1]
 [1]
 [0]]


## tensorflow
구글에서 개발한 오픈소스 머신러닝/ 딥러닝 프레임워크로 신경망 모델의 생성, 학습. 실행에 사용

In [ ]:
import tensorflow as tf
import numpy as np

X = tf.constant([[0., 0.], [0., 1.],[1., 0], [1., 1.]], dtype=tf.float32)

y = tf.constant([[0.], [1.], [1.], [0.]], dtype=tf.float32)

# 난수 고정
tf.random.set_seed(1)

# tf.Variable()
# 학습 과정에서 값이 변경되어야 하는 Tensor


W1 = tf.Variable(tf.random.normal([2, 4]), name="W1")

b1 = tf.Variable(tf.zeros([4]), name="b1")


W2 = tf.Variable(tf.random.normal([4, 1]), name="W2")

b2 = tf.Variable(tf.zeros([1]), name="b2")


learning_rate = 0.5



In [ ]:
# 학습
for epoch in range(10000):
  # GradientTape
  #Tensorflow 가 이 영역안에서 수행되는 계산과정을 기록
  #
  # 나중에 이 기록을 이용해서 자동으로 미분값을 계산
  with tf.GradientTape() as tape:
    #순전파
    #입력층 --> 은닉층
    hidden_input = ( tf.matmul(X, W1)+ b1)

    hidden_output = ( tf.sigmoid(hidden_input))

    # 은닉층 --> 출력
    final_input = ( tf.matmul(hidden_output, W2) + b2)

    output = tf.sigmoid(final_input)

    # 손실함수 ( MSE )
    loss = tf.reduce_mean(tf.square(y- output))
    # 자동 미분

    gradients = tape.gradient(loss, [W1, b1, W2, b2])

    # gradients[0] -> W1 의 gradient
    # gradients[1] -> b1 의 gradient
    # gradients[2] -> W2 의 gradient
    # gradients[3] -> b2 의 gradient

    # 가중치수정
    # W1.assign()  ==>  =
    # W1.assign_add ==> +=
    # W1.assing_sub ==> -=
    W1.assign_sub(learning_rate* gradients[0])
    b1.assign_sub(learning_rate* gradients[1])
    W2.assign_sub(learning_rate* gradients[2])
    b2.assign_sub(learning_rate* gradients[3])

    # 1000번 마다 한번씩 loss 출력
    if epoch % 1000 == 0:
      print(f"epoch : {epoch},  "
      f"loss : {loss.numpy():0.6f}"
      )




epoch : 0,  loss : 0.259114
epoch : 1000,  loss : 0.042068
epoch : 2000,  loss : 0.006227
epoch : 3000,  loss : 0.002959
epoch : 4000,  loss : 0.001884
epoch : 5000,  loss : 0.001364
epoch : 6000,  loss : 0.001062
epoch : 7000,  loss : 0.000865
epoch : 8000,  loss : 0.000727
epoch : 9000,  loss : 0.000626


In [ ]:
print(output.numpy())

[[0.01499344]
 [0.9743596 ]
 [0.97743195]
 [0.02829229]]


In [ ]:
# cast() 형변환 함수
pred = tf.cast(output >= 0.5, dtype=tf.int32)
pred

<tf.Tensor: shape=(4, 1), dtype=int32, numpy=
array([[0],
       [1],
       [1],
       [0]], dtype=int32)>

In [ ]:
print("최종 XOR 결과")
print(pred.numpy())

최종 XOR 결과
[[0]
 [1]
 [1]
 [0]]


## KERAS

딥러닝, 신경망 모델을 쉽고 빠르게 만들고 학습할수 있도록 제공되는 고수준 딥러닝 API

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

In [ ]:
X = np.array([[0,0], [0, 1], [1,0], [1, 1]],dtype=np.float32)
y = np.array([[0], [1], [1],[0]],dtype=np.float32 )

In [ ]:
X

array([[0., 0.],
       [0., 1.],
       [1., 0.],
       [1., 1.]], dtype=float32)

In [ ]:
# 신경망 모델 생성
# Sequential
# 신경망 층을 순서대로 쌓아가는 모델
model = keras.Sequential([
    # 입력정보
    # [x1, x2]  , 특징(feature) 2
    keras.Input(shape=(2,)),

    # 은닉층
    # Dense
    # 모든 입력 뉴런과 모든 출력 뉴런이 연결된 완전연결층
    keras.layers.Dense(units=4, activation="sigmoid"),
    # 출력층
    keras.layers.Dense(units=1, activation="sigmoid")
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 4)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17 (68.00 B)

 Trainable params: 17 (68.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 학습방법 설정

model.compile(
    # 가중치 를 수정하는 방법
    optimizer= keras.optimizers.Adam(learning_rate=0.05),
    # 손실함수  : 0/1 이진분류 문제
    loss = "binary_crossentropy",
    #학습과정 정확도 확인
    metrics = ['accuracy']
)



In [ ]:
model.fit(X, y, epochs=2000, verbose=1)

Epoch 1/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 879ms/step - accuracy: 0.5000 - loss: 0.7184
Epoch 2/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5000 - loss: 0.7024
Epoch 3/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 0.6948
Epoch 4/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 0.6945
Epoch 5/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5000 - loss: 0.6976
Epoch 6/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 0.6998
Epoch 7/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 0.6997
Epoch 8/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 0.6979
Epoch 9/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 0.6954
Epoch 10/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 0.6933
Epoch 11/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss: 0.6920
Epoch 12/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy

In [ ]:
prediction_probability = model.predict(X, verbose=0)
prediction_probability

array([[3.6209272e-05],
       [9.9940729e-01],
       [9.9982119e-01],
       [7.1786420e-04]], dtype=float32)

In [ ]:
print("신경망 출력값")
print(np.round(prediction_probability,2))

신경망 출력값
[[0.]
 [1.]
 [1.]
 [0.]]
